This notebook does the revisions the supervisors asked for on the Task 2 store comparison ("Do matched and unmatched stores differ?", step 9 Part C).
It uses the step 9 tables as they are: [9ci] (one row per storefront, the 542), [9ciii] (one row per open storefront-month, 2019-2021) and [9bi] (place
characteristics).

(a) A balance table of matched vs. unmatched storefronts: average sales, median opening year, opening duration, number of neighboring businesses, log
population per square mile in the place, and urban share. Label as ([10a])

(b) The share of storefronts and of storefront-months matched, by opening year. Label as ([10b])

(c) The share of storefronts and of storefront-months matched, by opening duration: <= 12, 13-24, and > 24 months. Label as ([10c])

(d) Linear probability models of matched (1) / unmatched (0) on opening year, opening duration, log (1 + sales), and log place density, in the eight
combinations asked for. Label as ([10d])

Definitions (the same as in step 9 unless said otherwise):
- **Stores with no sales are dropped** in (b)-(d), and in the main version of (a): storefronts with no positive sales in any of their open months in
  2019-2021 (the step 9 Part A sales gate). That drops 50 of the 542 storefronts (49 unmatched, 1 matched), leaving 492 (426 matched, 66 unmatched). (a)
  is also shown for all 542
- **Opening year**: the year of the storefront's first month with positive retail sales by any license operating there (LCB archive before 2018, the store
  history file from 2018; [9ci] `first_sales_month`). Every storefront kept has one. 2014 (from Jul 2014, when retail sales start) is the baseline
- **Opening duration**: months the storefront is open in 2019-2021 by the dated WSLCB samples ([9ci] `open_months`, out of 36), in three groups: <= 12
  (baseline), 13-24, > 24
- **Sales**: mean monthly net sales over the storefront's open months in 2019-2021, with months with no sales record counted as 0 ([9ci]
  `mean_monthly_sales`); log (1 + sales) in the regressions
- **Neighboring businesses**: SafeGraph POIs within 150 m (within 50 m as a check); current snapshot, not 2019-2021
- **Log population per square mile in the place** and **urban share**: the census place the storefront is in ([9bi]: 2020 Census population over TIGER land
  area; share of the place's people in 2020 urban blocks). Storefronts outside any census place (67 of the 492) have neither, so they drop out of those rows
  and of regressions (5), (6), (8). (d) is also run on the 425 storefronts in a place, so every column has the same sample
- A storefront-month is matched when it has an approved Veraset label that month; a storefront is matched when any of its open months is
- Tables also get saved to "Task 2/Charts and Figures" as .html files, next to the charts

In [1]:
### IMPORTS AND PRELIMINARIES
import os, warnings
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)

STOREFRONT = ["street_address_norm", "city_norm"]
CHART_FOLDER = "Task 2/Charts and Figures"
os.makedirs(CHART_FOLDER, exist_ok = True)
DURATIONS = ["<= 12 months", "13-24 months", "> 24 months"]

def show(fig, name):
    fig.show()
    fig.write_html(f"{CHART_FOLDER}/{name}.html", include_plotlyjs = "cdn")

TABLE_CSS = ("<style>body{font-family:Georgia,serif;margin:24px;color:#222} table{border-collapse:collapse;font-size:14px}"
             " th,td{padding:4px 12px;text-align:right;border:none} th{border-bottom:1px solid #222} td:first-child,th:first-child{text-align:left}"
             " tr:last-child td{border-bottom:1px solid #222} caption{caption-side:top;text-align:left;font-weight:bold;padding-bottom:8px}"
             " p{font-size:12px;max-width:900px}</style>")

def save_table(df, name, title, note = ""):
    html = df.to_html(index = False, escape = False, na_rep = "")
    html = html.replace('<table border="1" class="dataframe">', f"<table><caption>{title}</caption>", 1)
    with open(f"{CHART_FOLDER}/{name}.html", "w") as f:
        f.write(f"<!doctype html><html><head><meta charset='utf-8'><title>{title}</title>{TABLE_CSS}</head><body>{html}<p>{note}</p></body></html>")

In [2]:
### LOADING: step 9 tables
## Output: stores_all (542 storefronts) and stores (the 492 with sales), with the variables used below
stores_all = pd.read_parquet("processed-data/9ci_storefront_characteristics.parquet")
place_chars = pd.read_parquet("processed-data/9bi_place_match_rates.parquet")[["place_fips", "pop_density", "urban_share"]]
store_months_all = pd.read_parquet("processed-data/9ciii_storefront_months.parquet")

d = stores_all.merge(place_chars, on = "place_fips", how = "left")
d["any_sales"] = d.share_open_months_with_sales.fillna(0) > 0
d["opening_year"] = pd.to_numeric(d.first_sales_month.str[:4], errors = "coerce")
d["duration"] = pd.cut(d.open_months, [0, 12, 24, 36], labels = DURATIONS)
d["log_sales"] = np.log1p(d.mean_monthly_sales.fillna(0))
d["log_place_density"] = np.log(d.pop_density)
d["matched_i"] = d.matched.astype(int)
d["group"] = np.where(d.matched, "matched", "unmatched")
stores_all = d
stores = d[d.any_sales].copy()

print("storefronts:", len(stores_all), "| with sales in 2019-2021 (kept):", len(stores), "| dropped:", (~stores_all.any_sales).sum())
display(pd.crosstab(stores_all.any_sales, stores_all.group, margins = True))
print("kept storefronts with no opening year:", stores.opening_year.isna().sum(), "| outside any census place:", stores.place_fips.isna().sum())

storefronts: 542 | with sales in 2019-2021 (kept): 492 | dropped: 50


group,matched,unmatched,All
any_sales,,,
False,1,49,50
True,426,66,492
All,427,115,542


kept storefronts with no opening year: 0 | outside any census place: 67


### (a) Balance table

In [3]:
### (a) BALANCE TABLE
## Output: [10a] one row per characteristic and sample ("492 with sales", "all 542"): "matched_n", "unmatched_n", "matched_mean", "matched_sd",
##         "unmatched_mean", "unmatched_sd", "difference" (matched - unmatched), "p_value" (Welch t-test), "matched_median", "unmatched_median",
##         "ranksum_p" (Mann-Whitney)
## Name: 10a_balance_table.parquet
BALANCE = {"mean_monthly_sales": "Average monthly sales, 2019-2021 ($)",
           "log_sales": "log (1 + average monthly sales)",
           "opening_year": "Opening year (first month with sales)",
           "open_months": "Opening duration (months open, 2019-2021, of 36)",
           "dur_le12": "  Share open <= 12 months",
           "dur_13_24": "  Share open 13-24 months",
           "dur_gt24": "  Share open > 24 months",
           "pois_150m": "Neighboring businesses within 150 m",
           "pois_50m": "Neighboring businesses within 50 m",
           "log_place_density": "log population per sq. mile in the place",
           "urban_share": "Urban share of the place"}
for g, c in [("<= 12 months", "dur_le12"), ("13-24 months", "dur_13_24"), ("> 24 months", "dur_gt24")]:
    stores_all[c] = (stores_all.duration == g).astype(float)
stores = stores_all[stores_all.any_sales].copy()

def balance(data, sample):
    rows = []
    m, u = data[data.matched], data[~data.matched]
    for c, label in BALANCE.items():
        a, b = m[c].dropna().astype(float), u[c].dropna().astype(float)
        rows.append({"sample": sample, "characteristic": c, "label": label, "matched_n": len(a), "unmatched_n": len(b),
                     "matched_mean": a.mean(), "matched_sd": a.std(), "unmatched_mean": b.mean(), "unmatched_sd": b.std(),
                     "difference": a.mean() - b.mean(), "p_value": stats.ttest_ind(a, b, equal_var = False).pvalue,
                     "matched_median": a.median(), "unmatched_median": b.median(), "ranksum_p": stats.mannwhitneyu(a, b).pvalue})
    return pd.DataFrame(rows)

bal = pd.concat([balance(stores, "492 with sales"), balance(stores_all, "all 542")], ignore_index = True)
bal.to_parquet("processed-data/10a_balance_table.parquet")

def fmt(x, c):
    if pd.isna(x): return ""
    if c == "mean_monthly_sales": return f"{x:,.0f}"
    if c == "opening_year": return f"{x:.1f}" if x % 1 else f"{x:.0f}"
    if c.startswith("dur_") or c == "urban_share": return f"{x:.2f}"
    if c in ("open_months", "pois_150m", "pois_50m"): return f"{x:.1f}"
    return f"{x:.2f}"

def pretty(b):
    out = pd.DataFrame({"": b.label,
        "Matched: mean (SD)": [f"{fmt(r.matched_mean, r.characteristic)} ({fmt(r.matched_sd, r.characteristic)})" for r in b.itertuples()],
        "Matched: median": [fmt(r.matched_median, r.characteristic) if not r.characteristic.startswith("dur_") else "" for r in b.itertuples()],
        "Unmatched: mean (SD)": [f"{fmt(r.unmatched_mean, r.characteristic)} ({fmt(r.unmatched_sd, r.characteristic)})" for r in b.itertuples()],
        "Unmatched: median": [fmt(r.unmatched_median, r.characteristic) if not r.characteristic.startswith("dur_") else "" for r in b.itertuples()],
        "Difference": [fmt(r.difference, r.characteristic) for r in b.itertuples()],
        "p (t-test)": b.p_value.map(lambda p: f"{p:.3f}"),
        "p (rank-sum)": [f"{r.ranksum_p:.3f}" if not r.characteristic.startswith("dur_") else "" for r in b.itertuples()],
        "N (matched / unmatched)": [f"{r.matched_n} / {r.unmatched_n}" for r in b.itertuples()]})
    return out

NOTE_A = ("Storefronts are WSLCB addresses with a license operating in some month of 2019-2021; matched = an approved Veraset label in some open month. "
          "Sales: mean monthly net sales over the storefront's open months in 2019-2021 (months with no sales record = 0). Opening year: year of the first "
          "month with positive retail sales (LCB archive from Jul 2014, store history file from 2018). Opening duration: months open in 2019-2021 by the dated "
          "WSLCB samples. Neighboring businesses: SafeGraph POIs (current snapshot). Place density and urban share: the storefront's census place (2020 Census); "
          "storefronts outside any place are left out of those rows. p-values: Welch t-test (chi-square-equivalent for shares) and Mann-Whitney rank-sum.")
for sample, name in [("492 with sales", "balance_table_matched_vs_unmatched"), ("all 542", "balance_table_matched_vs_unmatched_all_542")]:
    t = pretty(bal[bal["sample"] == sample])
    print(f"\n=== Balance table, {sample}"); display(t)
    save_table(t, name, f"Matched vs. unmatched storefronts ({sample})",
               NOTE_A + (" Storefronts with no sales in any open month of 2019-2021 are dropped." if sample != "all 542" else
                         " All 542 storefronts; opening year is missing for the 43 with no sales record at all."))


=== Balance table, 492 with sales


,,Matched: mean (SD),Matched: median,Unmatched: mean (SD),Unmatched: median,Difference,p (t-test),p (rank-sum),N (matched / unmatched)
0,"Average monthly sales, 2019-2021 ($)","253,647 (216,630)","198,761","92,897 (117,752)","56,492","160,750",0.000,0.000,426 / 66
1,log (1 + average monthly sales),12.07 (1.00),12.20,10.05 (2.62),10.94,2.02,0.000,0.000,426 / 66
2,Opening year (first month with sales),2016.2 (1.6),2016,2017.8 (1.9),2018,-1.7,0.000,0.000,426 / 66
3,"Opening duration (months open, 2019-2021, of 36)",34.8 (4.2),36.0,26.3 (9.9),26.5,8.5,0.000,0.000,426 / 66
4,Share open <= 12 months,0.01 (0.12),,0.17 (0.38),,-0.15,0.002,,426 / 66
5,Share open 13-24 months,0.05 (0.23),,0.32 (0.47),,-0.26,0.000,,426 / 66
6,Share open > 24 months,0.93 (0.25),,0.52 (0.50),,0.42,0.000,,426 / 66
7,Neighboring businesses within 150 m,39.7 (44.7),24.0,35.3 (42.5),29.5,4.3,0.446,0.707,426 / 66
8,Neighboring businesses within 50 m,6.8 (10.8),4.0,5.7 (8.2),3.0,1.2,0.295,0.186,426 / 66
9,log population per sq. mile in the place,7.88 (0.89),8.09,7.64 (1.30),8.11,0.23,0.189,0.547,367 / 58



=== Balance table, all 542


,,Matched: mean (SD),Matched: median,Unmatched: mean (SD),Unmatched: median,Difference,p (t-test),p (rank-sum),N (matched / unmatched)
11,"Average monthly sales, 2019-2021 ($)","253,053 (216,724)","198,712","53,315 (100,173)",888,"199,738",0.000,0.000,427 / 115
12,log (1 + average monthly sales),12.04 (1.15),12.20,5.77 (5.37),6.79,6.27,0.000,0.000,427 / 115
13,Opening year (first month with sales),2016.2 (1.6),2016,2017.7 (1.8),2018,-1.5,0.000,0.000,426 / 73
14,"Opening duration (months open, 2019-2021, of 36)",34.7 (4.3),36.0,25.3 (10.2),24.0,9.4,0.000,0.000,427 / 115
15,Share open <= 12 months,0.01 (0.12),,0.24 (0.43),,-0.23,0.000,,427 / 115
16,Share open 13-24 months,0.06 (0.23),,0.29 (0.45),,-0.23,0.000,,427 / 115
17,Share open > 24 months,0.93 (0.26),,0.47 (0.50),,0.46,0.000,,427 / 115
18,Neighboring businesses within 150 m,39.6 (44.6),24.0,41.0 (67.1),26.0,-1.4,0.832,0.424,427 / 115
19,Neighboring businesses within 50 m,6.8 (10.8),4.0,5.7 (9.0),3.0,1.1,0.262,0.021,427 / 115
20,log population per sq. mile in the place,7.88 (0.89),8.09,7.73 (1.13),8.04,0.15,0.227,0.574,367 / 99


### (b) Share matched by opening year

In [4]:
### (b) SHARE MATCHED BY OPENING YEAR (storefronts with sales)
## Output: [10b] one row per opening year: "storefronts", "matched_storefronts", "share_storefronts_matched", "storefront_months" (open months, 2019-2021),
##         "matched_storefront_months", "share_storefront_months_matched", and a "total" row
## Name: 10b_match_by_opening_year.parquet
keep = stores[STOREFRONT + ["opening_year", "duration"]]
sm = store_months_all.merge(keep, on = STOREFRONT, how = "inner")
print("storefront-months kept:", len(sm), "of", len(store_months_all))

def match_shares(by):
    st = stores.groupby(by, observed = True).agg(storefronts = ("matched", "size"), matched_storefronts = ("matched", "sum"))
    mo = sm.groupby(by, observed = True).agg(storefront_months = ("covered", "size"), matched_storefront_months = ("covered", "sum"))
    t = st.join(mo).reset_index()
    tot = pd.DataFrame([{by: "total", **t.drop(columns = by).sum().to_dict()}])
    t = pd.concat([t.astype({by: str}), tot], ignore_index = True)
    t["share_storefronts_matched"] = t.matched_storefronts / t.storefronts
    t["share_storefront_months_matched"] = t.matched_storefront_months / t.storefront_months
    return t[[by, "storefronts", "matched_storefronts", "share_storefronts_matched", "storefront_months", "matched_storefront_months",
              "share_storefront_months_matched"]]

stores["opening_year"] = stores.opening_year.astype(int)
sm["opening_year"] = sm.opening_year.astype(int)
by_year = match_shares("opening_year")
by_year.to_parquet("processed-data/10b_match_by_opening_year.parquet")
display(by_year.round(3))

def pretty_shares(t, first):
    return pd.DataFrame({first: t.iloc[:, 0],
                         "Storefronts": t.storefronts.astype(int), "Matched": t.matched_storefronts.astype(int),
                         "Share matched": t.share_storefronts_matched.map(lambda x: f"{x:.1%}"),
                         "Storefront-months": t.storefront_months.astype(int), "Matched ": t.matched_storefront_months.astype(int),
                         "Share matched ": t.share_storefront_months_matched.map(lambda x: f"{x:.1%}")})
NOTE_B = ("Storefronts with positive sales in some open month of 2019-2021 (492). Storefront-months: the storefront's open months in 2019-2021 by the "
          "dated WSLCB samples; a storefront-month is matched when it has an approved Veraset label that month, a storefront when any of its months is.")
save_table(pretty_shares(by_year, "Opening year"), "match_share_by_opening_year_table", "Share of storefronts and storefront-months matched, by opening year",
           NOTE_B + " Opening year: year of the first month with positive retail sales (2014 = from Jul 2014).")

def share_chart(t, by, title, name, order):
    t = t[t[by] != "total"]
    long = pd.concat([t.assign(level = "storefronts", share = t.share_storefronts_matched, n = t.storefronts),
                      t.assign(level = "storefront-months", share = t.share_storefront_months_matched, n = t.storefront_months)])
    long["label"] = long.apply(lambda r: f"{r.share:.0%}<br>n={int(r.n)}", axis = 1)
    fig = px.bar(long, x = by, y = "share", color = "level", barmode = "group", text = "label", title = title,
                 category_orders = {by: order, "level": ["storefronts", "storefront-months"]}, labels = {"share": "share matched", "level": ""})
    fig.update_traces(textposition = "outside")
    fig.update_yaxes(range = [0, 1.15], tickformat = ".0%")
    show(fig, name)

share_chart(by_year, "opening_year", "Share matched by opening year (storefronts with sales, 2019-2021)", "match_share_by_opening_year",
            [str(y) for y in range(2014, 2022)])

storefront-months kept: 16542 of 17742


,opening_year,storefronts,matched_storefronts,share_storefronts_matched,storefront_months,matched_storefront_months,share_storefront_months_matched
0,2014,69,67,0.971,2403,1815,0.755
1,2015,96,88,0.917,3290,2382,0.724
2,2016,128,122,0.953,4403,3187,0.724
3,2017,77,68,0.883,2640,1764,0.668
4,2018,64,45,0.703,2137,1172,0.548
5,2019,25,17,0.680,851,429,0.504
6,2020,27,18,0.667,690,180,0.261
7,2021,6,1,0.167,128,4,0.031
8,total,492,426,0.866,16542,10933,0.661


### (c) Share matched by opening duration

In [5]:
### (c) SHARE MATCHED BY OPENING DURATION (storefronts with sales)
## Output: [10c] the same columns as [10b], one row per duration group (months open in 2019-2021: <= 12, 13-24, > 24) and a "total" row
## Name: 10c_match_by_duration.parquet
by_dur = match_shares("duration")
by_dur.to_parquet("processed-data/10c_match_by_duration.parquet")
display(by_dur.round(3))
save_table(pretty_shares(by_dur, "Months open, 2019-2021"), "match_share_by_opening_duration_table",
           "Share of storefronts and storefront-months matched, by opening duration", NOTE_B + " Opening duration: months open in 2019-2021, of 36.")
share_chart(by_dur, "duration", "Share matched by opening duration (months open in 2019-2021; storefronts with sales)", "match_share_by_opening_duration",
            DURATIONS)

,duration,storefronts,matched_storefronts,share_storefronts_matched,storefront_months,matched_storefront_months,share_storefront_months_matched
0,<= 12 months,17,6,0.353,204,71,0.348
1,13-24 months,44,23,0.523,940,260,0.277
2,> 24 months,431,397,0.921,15398,10602,0.689
3,total,492,426,0.866,16542,10933,0.661


### (d) Regressions

In [6]:
### (d) LINEAR PROBABILITY MODELS OF MATCHED (storefronts with sales)
## matched (1/0) on: (1) opening year indicators; (2) opening duration indicators (<= 12 months baseline); (3) opening year + duration; (4) log (1 + sales);
## (5) log population per sq. mile in the place; (6) (4) + (5); (7) log (1 + sales) + opening year + duration; (8) (7) + log place density.
## OLS with heteroskedasticity-robust (HC1) standard errors. Opening year baseline: 2014
## Output: [10d] one row per sample, model, and term: "sample", "model", "term", "coef", "se", "p", "n", "r2", "mean_dep"
## Name: 10d_regressions.parquet
YEAR = "C(opening_year, Treatment(2014))"
DUR = "C(duration, Treatment('<= 12 months'))"
MODELS = {1: [YEAR], 2: [DUR], 3: [YEAR, DUR], 4: ["log_sales"], 5: ["log_place_density"], 6: ["log_sales", "log_place_density"],
          7: ["log_sales", YEAR, DUR], 8: ["log_sales", "log_place_density", YEAR, DUR]}
reg = stores.assign(duration = stores.duration.astype(str))

fits, rows = {}, []
for sample, data in [("492 with sales", reg), ("425 with sales, in a census place", reg[reg.log_place_density.notna()])]:
    for k, terms in MODELS.items():
        dd = data.dropna(subset = ["log_place_density"]) if "log_place_density" in terms else data
        fit = smf.ols("matched_i ~ " + " + ".join(terms), data = dd).fit(cov_type = "HC1")
        fits[(sample, k)] = fit
        for term in fit.params.index:
            rows.append({"sample": sample, "model": k, "term": term, "coef": fit.params[term], "se": fit.bse[term], "p": fit.pvalues[term],
                         "n": int(fit.nobs), "r2": fit.rsquared, "mean_dep": dd.matched_i.mean()})
regs = pd.DataFrame(rows)
regs.to_parquet("processed-data/10d_regressions.parquet")

def term_label(t):
    if t.startswith("C(opening_year"): return "Opened " + t.split("[T.")[1].rstrip("]")
    if t.startswith("C(duration"): return "Open " + t.split("[T.")[1].rstrip("]")
    return {"log_sales": "log (1 + sales)", "log_place_density": "log population per sq. mile (place)", "Intercept": "Constant"}.get(t, t)

ORDER = (["log_sales", "log_place_density"] + [f"{YEAR}[T.{y}]" for y in range(2015, 2022)]
         + [f"{DUR}[T.13-24 months]", f"{DUR}[T.> 24 months]", "Intercept"])
stars = lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""

def reg_table(sample):
    out = []
    for t in ORDER:
        coef_row, se_row = {"": term_label(t)}, {"": ""}
        for k in MODELS:
            f = fits[(sample, k)]
            coef_row[f"({k})"] = f"{f.params[t]:.3f}{stars(f.pvalues[t])}" if t in f.params else ""
            se_row[f"({k})"] = f"({f.bse[t]:.3f})" if t in f.params else ""
        out += [coef_row, se_row]
    for label, fn in [("Opening year indicators", lambda k: "Yes" if YEAR in MODELS[k] else ""),
                      ("Opening duration indicators", lambda k: "Yes" if DUR in MODELS[k] else ""),
                      ("Observations", lambda k: f"{int(fits[(sample, k)].nobs)}"),
                      ("R-squared", lambda k: f"{fits[(sample, k)].rsquared:.3f}"),
                      ("Mean of matched", lambda k: f"{fits[(sample, k)].model.endog.mean():.3f}")]:
        out.append({"": label, **{f"({k})": fn(k) for k in MODELS}})
    return pd.DataFrame(out)

NOTE_D = ("Linear probability models; dependent variable = 1 if the storefront is matched (an approved Veraset label in some open month of 2019-2021), 0 if "
          "not. Storefronts with no positive sales in any open month of 2019-2021 are dropped. Opening year: year of the first month with positive retail "
          "sales (baseline 2014). Opening duration: months open in 2019-2021 (baseline <= 12). Sales: mean monthly net sales over open months, 2019-2021. "
          "Place density: 2020 population per square mile of land in the storefront's census place; storefronts outside any place drop out of the columns "
          "that use it. Heteroskedasticity-robust (HC1) standard errors in parentheses. * p < 0.1, ** p < 0.05, *** p < 0.01.")
for sample, name in [("492 with sales", "regressions_matched"), ("425 with sales, in a census place", "regressions_matched_place_sample")]:
    t = reg_table(sample)
    print(f"\n=== {sample}"); display(t)
    save_table(t, name, f"Which storefronts are matched? Linear probability models ({sample})",
               NOTE_D + (" All columns use the 425 storefronts in a census place." if "place" in sample else ""))

## Joint tests: are the opening year / duration indicators jointly significant?
for k in [1, 2, 3, 7, 8]:
    f = fits[("492 with sales", k)]
    for name, key in [("opening year", "C(opening_year"), ("duration", "C(duration")]:
        cols = [c for c in f.params.index if c.startswith(key)]
        if cols:
            w = f.wald_test(" = 0, ".join(cols) + " = 0", scalar = True, use_f = True)
            print(f"model ({k}) {name} indicators jointly zero: F = {float(w.statistic):.2f}, p = {float(w.pvalue):.4f}")


=== 492 with sales


,,(1),(2),(3),(4),(5),(6),(7),(8)
0,log (1 + sales),,,,0.105***,,0.109***,0.069***,0.061***
1,,,,,(0.009),,(0.010),(0.011),(0.012)
2,log population per sq. mile (place),,,,,0.030,-0.005,,0.007
3,,,,,,(0.021),(0.019),,(0.017)
4,Opened 2015,-0.054,,-0.045,,,,-0.033,-0.044
5,,(0.035),,(0.033),,,,(0.032),(0.031)
6,Opened 2016,-0.018,,-0.006,,,,0.007,-0.004
7,,(0.028),,(0.024),,,,(0.024),(0.020)
8,Opened 2017,-0.088**,,-0.083**,,,,-0.066*,-0.068**
9,,(0.042),,(0.039),,,,(0.036),(0.033)



=== 425 with sales, in a census place


,,(1),(2),(3),(4),(5),(6),(7),(8)
0,log (1 + sales),,,,0.108***,,0.109***,0.062***,0.061***
1,,,,,(0.011),,(0.010),(0.012),(0.012)
2,log population per sq. mile (place),,,,,0.030,-0.005,,0.007
3,,,,,,(0.021),(0.019),,(0.017)
4,Opened 2015,-0.075**,,-0.056*,,,,-0.043,-0.044
5,,(0.035),,(0.031),,,,(0.031),(0.031)
6,Opened 2016,-0.030,,-0.011,,,,-0.001,-0.004
7,,(0.026),,(0.020),,,,(0.019),(0.020)
8,Opened 2017,-0.079*,,-0.079**,,,,-0.068**,-0.068**
9,,(0.041),,(0.034),,,,(0.033),(0.033)


model (1) opening year indicators jointly zero: F = 8.96, p = 0.0000
model (2) duration indicators jointly zero: F = 24.83, p = 0.0000
model (3) opening year indicators jointly zero: F = 6.51, p = 0.0000
model (3) duration indicators jointly zero: F = 18.75, p = 0.0000
model (7) opening year indicators jointly zero: F = 4.44, p = 0.0001
model (7) duration indicators jointly zero: F = 8.24, p = 0.0003
model (8) opening year indicators jointly zero: F = 4.60, p = 0.0001
model (8) duration indicators jointly zero: F = 8.92, p = 0.0002


### NOTES
(Written from the outputs above as of the last run. If the inputs change, re-check the numbers.)

**Sample**: dropping storefronts with no sales in any open month of 2019-2021 leaves 492 of the 542 (426 matched, 66 unmatched; 49 of the 50 dropped are
unmatched). Every one of the 492 has an opening year; 67 are outside any census place.

**(a) Balance table** (492 with sales; the all-542 version is in the second table)
- **Sales**: mean monthly sales $254k matched vs. $93k unmatched (medians $199k vs. $56k), p < 0.001. With all 542 the unmatched median is $888, because
  the 49 no-sales storefronts are in it.
- **Opening year**: median 2016 matched vs. 2018 unmatched (means 2016.2 vs. 2017.8), p < 0.001.
- **Opening duration**: 34.8 vs. 26.3 months open of 36 (medians 36 vs. 26.5), p < 0.001. 93% of matched storefronts are open more than 24 months against
  52% of unmatched; 17% of unmatched are open 12 months or less against 1% of matched.
- **Neighboring businesses**: no difference (SafeGraph POIs within 150 m: mean 39.7 vs. 35.3, median 24 vs. 29.5, p = 0.45; within 50 m p = 0.30).
- **log population per sq. mile in the place**: no difference (7.88 vs. 7.64, p = 0.19; rank-sum p = 0.55).
- **Urban share of the place**: lower for unmatched on average (0.91 vs. 0.74, p = 0.006), but both medians are 0.999 (rank-sum p = 0.06): a minority of
  unmatched storefronts are in mostly rural places.

**(b) By opening year**: the share of storefronts matched is 88-97% for stores that opened in 2014-2017, then 70% (2018), 68% (2019), 67% (2020) and 1 of 6
(2021). Storefront-months fall faster: 76% (2014), 72% (2015-2016), 67% (2017), 55% (2018), 50% (2019), 26% (2020), 3% (2021). Newer stores are less often
matched at all, and when matched, fewer of their months are.

**(c) By opening duration**: 35% of storefronts open 12 months or less are matched (6 of 17), 52% of those open 13-24 months (23 of 44), and 92% of those
open more than 24 months (397 of 431). Storefront-months: 35%, 28%, 69%.

**(d) Regressions** (linear probability, robust SEs, 492 storefronts; 425 when place density is in)
- (1) Opening year alone: stores opened 2014-2016 are matched at about the same rate (97% baseline; 2015 and 2016 not different from 2014); 2017 is 9 points
  lower, 2018-2020 27-30 points lower, 2021 80 points lower. R2 = 0.15.
- (2) Duration alone: > 24 months is 57 points more likely matched than <= 12 (p < 0.001); 13-24 months is +17 points, not significant. R2 = 0.19.
- (3) Both: each stays jointly significant (F-tests p < 0.001); the 2020 coefficient shrinks to -0.09 (insignificant) once duration is in, since 2020
  openers are mechanically open fewer months. R2 = 0.29.
- (4) log (1 + sales) alone: +0.105 per log point (p < 0.001). R2 = 0.21.
- (5) log place density alone: +0.030 (se 0.021), not significant. R2 = 0.007.
- (6) Sales and place density: sales +0.109, density -0.005 (insignificant).
- (7) Sales, opening year, duration: all three matter. Sales +0.069 per log point; 2018, 2019, 2021 openers 19-50 points less likely matched; > 24 months
  +33 points. R2 = 0.36.
- (8) Adding place density to (7): density stays at zero (+0.007, se 0.017); sales +0.061, opening year and duration about the same. R2 = 0.38.
- On the common sample of 425 storefronts in a place, the coefficients are close to the full-sample ones.

**Takeaway**: among storefronts that sold anything in 2019-2021, matching is explained by when a store opened, how long it was open, and how much it sold.
The place it is in (density) and the number of businesses around it don't predict matching; urban share differs only through a few rural unmatched stores.